# 16. 가이드북 준지도 학습 그대로 재현 + 누수 요인 점검

가이드북 2.3.2 (pseudo-labeling, 확신도 상위 10%씩, unlabeled 90% 쓸 때까지) **로직 그대로**.
가이드북 [표 4] RG3: RF F1 0.95 / ROC-AUC 0.98, DNN F1 0.94 — 이 수치가 재현되는지, 그리고 아래 두 가지를 하나씩 바꾸면 어떻게 되는지 확인

| 요인 | 가이드북 | 바꾼 버전 | 왜 |
|---|---|---|---|
| 분할 | 행 단위 | 샷 단위 | LH·RH 두 행은 공정값이 완전히 같음 → 평가 행의 쌍둥이가 학습에 들어감 |
| unlabeled | 그대로 | labeled 샷 제거 | 2차 labeled 샷이 unlabeled에도 있음 → 평가 샷이 pseudo-label 학습에 들어감 |

분할 seed를 여러 개 돌려 평가 불량 5~8개에서 오는 흔들림도 확인 (가이드북은 1회 분할)

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
from guidebook_semi import run_grid
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
OUT = './outputs/guidebook_semi'
os.makedirs(OUT, exist_ok=True)

UNL_SAMPLE = 5000          # 가이드북은 전체 사용. SVM이 느려서 기존 노트북과 같이 5000 (None = 전체)
# (A) 가이드북 그대로: seed 42, 모델 3종, RF 1200그루  — SVM이 느림 (수십 분)
A_SEEDS, A_MODELS, A_TREES = [42], ['SVM', 'RF', 'GNB'], 1200
# (B) 흔들림 확인: seed 5개, RF·GNB, RF 300그루 (속도용, 결론엔 영향 작음)
B_SEEDS, B_MODELS, B_TREES = [42, 0, 1, 2, 3], ['RF', 'GNB'], 300

## 1. 실행 — (A) 가이드북 그대로 + (B) seed 흔들림

In [ ]:
RES_A, _ = run_grid(models=A_MODELS, seeds=A_SEEDS, unl_sample=UNL_SAMPLE, rf_trees=A_TREES)
RES_B, _ = run_grid(models=B_MODELS, seeds=B_SEEDS, unl_sample=UNL_SAMPLE, rf_trees=B_TREES)
RES_A.to_csv(f'{OUT}/results_A_faithful.csv', index=False, encoding='utf-8-sig')
RES_B.to_csv(f'{OUT}/results_B_seeds.csv', index=False, encoding='utf-8-sig')

## 2. 가이드북과 같은 조건 (행 단위 분할 + unlabeled 그대로, seed 42)

In [ ]:
g = RES_A[(RES_A['분할'] == 'guidebook') & (RES_A['unlabeled 중복'] == 'keep')]
print('(A) 가이드북 조건'); 
display(g[['제품', '모델', '단계', '정확도', '정밀도', '재현율', 'ROC-AUC(0/1)', 'F1', 'PR-AUC(확률)', 'TP', 'FP', 'FN',
           'pseudo 불량', '쌍둥이 누수(test 행 중 train에 같은 공정값)', '사용 unlabeled 중 labeled 샷 행']].round(3))
print('(A) 요인별 — 분할·unlabeled를 바꾸면'); 
display(RES_A[RES_A['단계'] == '준지도(최종)'][['제품', '분할', 'unlabeled 중복', '모델', '재현율', '정밀도', 'F1', 'PR-AUC(확률)', 'pseudo 불량',
        '쌍둥이 누수(test 행 중 train에 같은 공정값)', '사용 unlabeled 중 labeled 샷 행']].round(3))

## 3. (B) 요인별 비교 — seed 5개 평균 ± 범위

In [ ]:
agg = (RES_B.groupby(['제품', '분할', 'unlabeled 중복', '모델', '단계'])[['F1', '재현율', '정밀도', 'PR-AUC(확률)', 'pseudo 불량']]
         .agg(['mean', 'min', 'max']).round(3))
agg.to_csv(f'{OUT}/summary_by_factor.csv', encoding='utf-8-sig')
display(agg)

## 4. (B) 준지도 효과 (최종 − 라벨만) — 같은 seed끼리

In [ ]:
k = ['제품', '분할', 'unlabeled 중복', 'seed', '모델']
w = RES_B.pivot_table(index=k, columns='단계', values=['F1', 'PR-AUC(확률)'])
eff = pd.DataFrame({'ΔF1': w[('F1', '준지도(최종)')] - w[('F1', '라벨만(반복0)')],
                    'ΔPR-AUC': w[('PR-AUC(확률)', '준지도(최종)')] - w[('PR-AUC(확률)', '라벨만(반복0)')]})
display(eff.groupby(level=[0, 1, 2, 4]).agg(['mean', 'min', 'max']).round(3))

## 해석 가이드
- **pseudo 불량 = 0**이면 확신도 상위가 전부 양품 → 불량 정보가 추가되지 않으니 준지도 효과가 구조적으로 없음
- 가이드북 조건에서만 높고, 샷 분할·중복 제거 후 떨어지면 → **가이드북 수치는 누수의 효과**
- seed마다 F1 범위가 크면 → 평가 불량 5~8개로는 단일 분할 결과를 믿기 어려움 (우리가 LORO + CI를 쓰는 이유)
- 비교 기준: 우리 LORO 결과 (13번 E2_phys, 14번 S0~S3) — 평가 방식이 달라 수치를 직접 비교하진 말고 '결론 방향'만 비교